In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned district-level data
df = pd.read_csv('/workspaces/Major_project/data/processed/district_level_data.csv')

print("Shape:", df.shape)
df = df.sort_values(['state_name', 'district_name', 'year']).reset_index(drop=True)


Shape: (5104, 51)


In [39]:
id_cols = ['id', 'year', 'state_name', 'state_code',
           'district_name', 'district_code', 'registration_circles']
crime_cols = [c for c in df.columns
              if c not in id_cols
              and c not in ['severity_score', 'total_crimes', 'risk_level']]

# Compute next year's severity per district (this is the TARGET)
df['next_year_severity'] = df.groupby(['state_name', 'district_name'])['severity_score'].shift(-1)

# Drop rows where we don't know next year's value (the last year of each district)
df = df.dropna(subset=['next_year_severity'])

# Create target from next year's severity
def classify_next_risk(x):
    if x >= df['next_year_severity'].quantile(0.75):
        return 'High'
    elif x >= df['next_year_severity'].quantile(0.25):
        return 'Medium'
    else:
        return 'Low'

df['target'] = df['next_year_severity'].apply(classify_next_risk)

print("Target distribution:")
print(df['target'].value_counts())
print(f"\nTotal rows after shift: {len(df)}")

Target distribution:
target
Medium    2142
High      1072
Low       1071
Name: count, dtype: int64

Total rows after shift: 4285


In [40]:
from sklearn.model_selection import train_test_split
# Features: THIS year's crime columns
X = df[crime_cols]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [41]:

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n=== ACCURACY ===")
print(f"{accuracy_score(y_test, y_pred):.4f}")

print("\n=== F1 SCORE (weighted) ===")
print(f"{f1_score(y_test, y_pred, average='weighted'):.4f}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))


=== ACCURACY ===
0.8856

=== F1 SCORE (weighted) ===
0.8860

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

        High       0.85      0.87      0.86       215
         Low       0.94      0.88      0.91       214
      Medium       0.88      0.89      0.89       428

    accuracy                           0.89       857
   macro avg       0.89      0.88      0.89       857
weighted avg       0.89      0.89      0.89       857



In [42]:
import joblib

joblib.dump(model, '../models/risk_model.pkl')
joblib.dump(crime_cols, '../models/feature_cols.pkl')

['../models/feature_cols.pkl']